# TelecomNexus – Data Pipeline

This notebook implements the Bronze, Silver and Gold data pipeline using PySpark and Delta Lake in Databricks. It ingests, cleans, validates and prepares telecom datasets for analytics.

In [0]:

import os

print("Current directory:")
print(os.getcwd())

print("\nFiles and folders visible here:")
print(os.listdir("."))


## Bronze ingestion

Read the 11 raw CSV datasets and store them as Delta tables in the Bronze schema.

In [0]:

import os

print("Repository folders:", os.listdir(".."))

print("\nBronze folders:", os.listdir("../data/bronze"))

print("\nTRAI QoS files:",
      os.listdir("../data/bronze/trai_qos"))

print("\nIBM files:",
      os.listdir("../data/bronze/ibm_telco"))


In [0]:

import os

repo_path = os.path.dirname(os.getcwd())
bronze_path = os.path.join(repo_path, "data", "bronze")

print("CSV FILES FOUND:\n")

csv_files = []

for root, dirs, files in os.walk(bronze_path):
    for file in files:
        if file.endswith(".csv"):
            path = os.path.join(root, file)
            csv_files.append(path)
            print(path)

print("\nTotal CSV files:", len(csv_files))

print("\nAVAILABLE CATALOGS:")
spark.sql("SHOW CATALOGS").show(truncate=False)

print("\nCURRENT CATALOG:")
spark.sql("SELECT current_catalog(), current_schema()").show()


In [0]:

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Schemas created:")
spark.sql("SHOW SCHEMAS IN workspace").show()


In [0]:

qos_path = "file:/Workspace/Users/2501731298@krmu.edu.in/Telecom-Project/data/bronze/trai_qos/trai_qos_june_2026.csv"

qos_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .load(qos_path)
)

print("Rows:", qos_df.count())
print("Columns:", len(qos_df.columns))

qos_df.printSchema()
qos_df.show(5, truncate=False)

(qos_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.trai_qos"))
    
print("Bronze QoS table created!")


In [0]:
spark.sql("SELECT COUNT(*) FROM workspace.bronze.trai_qos").show()

In [0]:

import re

base = "file:/Workspace/Users/2501731298@krmu.edu.in/Telecom-Project/data/bronze"

datasets = [
    ("ibm_telco_customer_churn", "ibm_telco/ibm_telco_customer_churn.csv"),
    ("trai_circle_wise_growth_rates", "trai_subscribers/circle_wise_growth_rates.csv"),
    ("trai_circle_wise_net_additions", "trai_subscribers/circle_wise_net_additions.csv"),
    ("trai_fixed_wired_broadband", "trai_subscribers/fixed_wired_broadband.csv"),
    ("trai_mnp_service_area_status", "trai_subscribers/mnp_service_area_status.csv"),
    ("trai_telephone_subscriber_base", "trai_subscribers/telephone_subscriber_base.csv"),
    ("trai_wireless_broadband", "trai_subscribers/wireless_broadband.csv"),
    ("trai_wireless_mobile_subscribers", "trai_subscribers/wireless_mobile_subscribers.csv"),
    ("trai_usage_by_service_area", "trai_usage/usage_by_service_area.csv"),
    ("trai_wireless_usage_trends", "trai_usage/wireless_usage_trends.csv")
]

results = []

for table_name, relative_path in datasets:
    path = f"{base}/{relative_path}"

    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .load(path)
    )

    # Clean column names for Delta compatibility
    for old_name in df.columns:
        new_name = re.sub(r"[^A-Za-z0-9_]", "_", old_name)
        new_name = re.sub(r"_+", "_", new_name).strip("_")
        df = df.withColumnRenamed(old_name, new_name)

    row_count = df.count()

    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(f"workspace.bronze.{table_name}"))

    results.append((table_name, row_count, len(df.columns)))
    print(f"Created: {table_name} | Rows: {row_count} | Columns: {len(df.columns)}")

print("\nBRONZE INGESTION COMPLETED")


## Bronze validation

Verify that all 11 Bronze tables were created successfully and contain the expected number of rows.

In [0]:

tables = [
    "trai_qos",
    "ibm_telco_customer_churn",
    "trai_circle_wise_growth_rates",
    "trai_circle_wise_net_additions",
    "trai_fixed_wired_broadband",
    "trai_mnp_service_area_status",
    "trai_telephone_subscriber_base",
    "trai_wireless_broadband",
    "trai_wireless_mobile_subscribers",
    "trai_usage_by_service_area",
    "trai_wireless_usage_trends"
]

for table in tables:
    print(
        table,
        spark.table(f"workspace.bronze.{table}").count()
    )


## IBM Silver transformation

Clean the IBM Telco customer churn dataset, standardise values, convert numeric columns and handle blank values.

In [0]:

from pyspark.sql.functions import col, trim, when

ibm_df = spark.table("workspace.bronze.ibm_telco_customer_churn")

# Trim whitespace from string columns
for c in ibm_df.columns:
    ibm_df = ibm_df.withColumn(c, trim(col(c)))

# Convert blank Total Charges into NULL
ibm_df = ibm_df.withColumn(
    "Total_Charges",
    when(col("Total_Charges") == "", None)
    .otherwise(col("Total_Charges"))
)

# Cast only genuinely numeric columns
numeric_columns = {
    "Senior_Citizen": "int",
    "Tenure_Months": "int",
    "Monthly_Charges": "double",
    "Total_Charges": "double",
    "Churn_Value": "int",
    "Churn_Score": "int"
}

for column_name, data_type in numeric_columns.items():
    if column_name in ibm_df.columns:
        ibm_df = ibm_df.withColumn(
            column_name,
            col(column_name).cast(data_type)
        )

ibm_df.printSchema()


In [0]:
print("Rows:", ibm_df.count())
ibm_df.select("Total_Charges", "Churn_Label", "Churn_Value").show(10)

In [0]:
ibm_df.printSchema()

for c in ibm_df.columns:
    print(c, ibm_df.schema[c].dataType)

In [0]:

from pyspark.sql import functions as F

# Start fresh from Bronze
ibm_clean = spark.table("workspace.bronze.ibm_telco_customer_churn")

# Trim all string columns
for field in ibm_clean.schema.fields:
    if field.dataType.simpleString() == "string":
        ibm_clean = ibm_clean.withColumn(
            field.name, F.trim(F.col(field.name))
        )

# Convert blank Total Charges to NULL
ibm_clean = ibm_clean.withColumn(
    "Total_Charges",
    F.when(
        F.col("Total_Charges") == "",
        None
    ).otherwise(F.col("Total_Charges"))
)

# Safely convert ONLY numeric columns
numeric_columns = {
    "Senior_Citizen": "INT",
    "Tenure_Months": "INT",
    "Monthly_Charges": "DOUBLE",
    "Total_Charges": "DOUBLE",
    "Churn_Value": "INT",
    "Churn_Score": "INT"
}

for name, dtype in numeric_columns.items():
    ibm_clean = ibm_clean.withColumn(
        name,
        F.expr(f"try_cast(`{name}` AS {dtype})")
    )

# Check before saving
print("Rows:", ibm_clean.count())
ibm_clean.printSchema()

# Save with schema replacement
(ibm_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.ibm_telco_customer_churn"))

print("IBM Silver table saved successfully!")


In [0]:

silver = spark.table("workspace.silver.ibm_telco_customer_churn")

print("Rows:", silver.count())
print("Unique customers:", silver.select("CustomerID").distinct().count())

silver.groupBy("Churn_Label").count().show()

silver.select(
    F.count(F.when(F.col("Total_Charges").isNull(), 1))
    .alias("null_total_charges")
).show()


## TRAI Silver transformation

Clean the TRAI QoS, MNP, service-area usage and national wireless usage datasets. Standardise LSA names using the provided metadata crosswalk.

In [0]:

trai_tables = [
    "trai_qos",
    "trai_mnp_service_area_status",
    "trai_usage_by_service_area",
    "trai_wireless_usage_trends"
]

for name in trai_tables:
    print("\n==========", name, "==========")
    df = spark.table(f"workspace.bronze.{name}")
    df.printSchema()
    df.show(2, truncate=False)

print("\n========== METADATA FILES ==========")
import os

repo_path = os.path.dirname(os.getcwd())
metadata_path = os.path.join(repo_path, "metadata")

print(os.listdir(metadata_path))


In [0]:
metadata_path = os.path.join(repo_path, "metadata", "dim_lsa.csv")

lsa_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv("file:" + metadata_path)
)

lsa_df.printSchema()
lsa_df.show(30, truncate=False)

In [0]:

from pyspark.sql import functions as F

# 1. QoS Silver
qos = spark.table("workspace.bronze.trai_qos")

qos = qos.withColumn(
    "serial_number",
    F.expr("try_cast(serial_number AS INT)")
)

for i in range(1, 14):
    c = f"metric_{i:02d}"
    qos = qos.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))

qos_map = lsa_df.select(
    "canonical_lsa",
    F.col("qos_lsa").alias("lsa")
)

qos_silver = qos.join(qos_map, "lsa", "left")

# 2. MNP Silver
mnp = spark.table("workspace.bronze.trai_mnp_service_area_status")

for c in [
    "may_2026_porting_requests_millions",
    "june_2026_porting_requests_millions"
]:
    mnp = mnp.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))

mnp_map = lsa_df.select(
    "canonical_lsa",
    F.col("mnp_service_area").alias("service_area")
)

mnp_silver = mnp.join(mnp_map, "service_area", "left")

# 3. Usage by service area Silver
usage = spark.table("workspace.bronze.trai_usage_by_service_area")

usage = usage.withColumn(
    "prepaid_percentage",
    F.regexp_replace("prepaid_percentage", "%", "")
)

usage = usage.withColumn(
    "prepaid_percentage",
    F.expr("try_cast(prepaid_percentage AS DOUBLE)")
)

usage_numeric = [
    "arpu_inr_per_month",
    "outgoing_sms_per_month",
    "outgoing_mou_per_month",
    "incoming_mou_per_month",
    "total_mou_per_month"
]

for c in usage_numeric:
    usage = usage.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))

usage_map = lsa_df.select(
    "canonical_lsa",
    F.col("usage_service_area").alias("service_area")
)

usage_silver = usage.join(usage_map, "service_area", "left")

# 4. National wireless usage trends Silver
trends = spark.table("workspace.bronze.trai_wireless_usage_trends")

trend_numeric = [
    "arpu_inr_per_month",
    "revenue_realization_per_gb_inr",
    "mou_per_subscriber",
    "local_outgoing_mou",
    "nld_outgoing_mou",
    "ild_outgoing_mou",
    "outgoing_sms_per_subscriber"
]

for c in trend_numeric:
    trends = trends.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))

# Save all four Silver tables
outputs = [
    (qos_silver, "trai_qos"),
    (mnp_silver, "trai_mnp_service_area_status"),
    (usage_silver, "trai_usage_by_service_area"),
    (trends, "trai_wireless_usage_trends")
]

for df, name in outputs:
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.silver.{name}"))
    print("Saved:", name, "| Rows:", df.count())

print("Four TRAI Silver tables created!")


In [0]:
for name in [
    "trai_qos",
    "trai_mnp_service_area_status",
    "trai_usage_by_service_area",
    "trai_wireless_usage_trends"
]:
    df = spark.table(f"workspace.silver.{name}")
    print(name, "Rows:", df.count())
    print("Unmapped LSA rows:",
          df.filter(F.col("canonical_lsa").isNull()).count()
          if "canonical_lsa" in df.columns else "Not applicable")

## Subscriber Silver transformation

Clean and convert the six TRAI subscriber datasets into Silver Delta tables.

In [0]:

from pyspark.sql import functions as F

remaining_tables = [
    "trai_circle_wise_growth_rates",
    "trai_circle_wise_net_additions",
    "trai_fixed_wired_broadband",
    "trai_telephone_subscriber_base",
    "trai_wireless_broadband",
    "trai_wireless_mobile_subscribers"
]

for name in remaining_tables:

    df = spark.table(f"workspace.bronze.{name}")

    # Safely convert numeric-looking columns, preserving text categories
    for field in df.schema.fields:
        c = field.name

        if c.lower() in ["serial_number", "s_no", "sr_no"]:
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS INT)"))

        elif any(word in c.lower() for word in [
            "subscriber", "growth", "addition", "percentage",
            "penetration", "rate", "number", "population"
        ]):
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.silver.{name}"))

    print(name, "saved | Rows:", df.count())

print("All six subscriber Silver tables created!")


In [0]:

silver_tables = [
    "ibm_telco_customer_churn",
    "trai_qos",
    "trai_mnp_service_area_status",
    "trai_usage_by_service_area",
    "trai_wireless_usage_trends",
    "trai_circle_wise_growth_rates",
    "trai_circle_wise_net_additions",
    "trai_fixed_wired_broadband",
    "trai_telephone_subscriber_base",
    "trai_wireless_broadband",
    "trai_wireless_mobile_subscribers"
]

for name in silver_tables:
    print(name, spark.table(f"workspace.silver.{name}").count())

print("Total Silver tables:", len(silver_tables))


## Gold analytics

Create aggregated Gold tables for customer churn analysis and TRAI network switching and usage context.

In [0]:

from pyspark.sql import functions as F

ibm = spark.table("workspace.silver.ibm_telco_customer_churn")

# 1. Overall churn overview
overview = ibm.agg(
    F.count("*").alias("total_customers"),
    F.sum(F.when(F.col("Churn_Value") == 1, 1).otherwise(0)).alias("churned_customers"),
    F.sum(F.when(F.col("Churn_Value") == 0, 1).otherwise(0)).alias("retained_customers"),
    F.round(
        F.avg(F.when(F.col("Churn_Value") == 1, 1.0).otherwise(0.0)) * 100, 2
    ).alias("churn_rate_percent"),
    F.round(F.avg("Monthly_Charges"), 2).alias("avg_monthly_charges"),
    F.round(F.avg("Tenure_Months"), 2).alias("avg_tenure_months")
)

# 2. Churn by contract
by_contract = ibm.groupBy("Contract").agg(
    F.count("*").alias("total_customers"),
    F.sum("Churn_Value").alias("churned_customers")
).withColumn(
    "churn_rate_percent",
    F.round(F.col("churned_customers") / F.col("total_customers") * 100, 2)
)

# 3. Churn by payment method
by_payment = ibm.groupBy("Payment_Method").agg(
    F.count("*").alias("total_customers"),
    F.sum("Churn_Value").alias("churned_customers")
).withColumn(
    "churn_rate_percent",
    F.round(F.col("churned_customers") / F.col("total_customers") * 100, 2)
)

# 4. Churn by tenure bands
tenure = ibm.withColumn(
    "tenure_band",
    F.when(F.col("Tenure_Months") <= 12, "0-12 months")
     .when(F.col("Tenure_Months") <= 24, "13-24 months")
     .when(F.col("Tenure_Months") <= 48, "25-48 months")
     .otherwise("49+ months")
)

by_tenure = tenure.groupBy("tenure_band").agg(
    F.count("*").alias("total_customers"),
    F.sum("Churn_Value").alias("churned_customers")
).withColumn(
    "churn_rate_percent",
    F.round(F.col("churned_customers") / F.col("total_customers") * 100, 2)
)

# 5. Churn by internet service
by_service = ibm.groupBy("Internet_Service").agg(
    F.count("*").alias("total_customers"),
    F.sum("Churn_Value").alias("churned_customers")
).withColumn(
    "churn_rate_percent",
    F.round(F.col("churned_customers") / F.col("total_customers") * 100, 2)
)

gold_outputs = [
    (overview, "ibm_churn_overview"),
    (by_contract, "ibm_churn_by_contract"),
    (by_payment, "ibm_churn_by_payment_method"),
    (by_tenure, "ibm_churn_by_tenure"),
    (by_service, "ibm_churn_by_service")
]

for df, name in gold_outputs:
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.gold.{name}"))
    print("Created:", name)

print("IBM Gold tables completed!")


In [0]:

mnp = spark.table("workspace.silver.trai_mnp_service_area_status").alias("m")
usage = spark.table("workspace.silver.trai_usage_by_service_area").alias("u")

trai_context = mnp.join(
    usage,
    F.col("m.canonical_lsa") == F.col("u.canonical_lsa"),
    "inner"
).select(
    F.col("m.canonical_lsa"),
    F.col("m.zone"),
    F.col("m.report_month").alias("mnp_report_month"),
    F.col("m.may_2026_porting_requests_millions"),
    F.col("m.june_2026_porting_requests_millions"),
    F.col("u.category"),
    F.col("u.prepaid_percentage"),
    F.col("u.arpu_inr_per_month"),
    F.col("u.outgoing_mou_per_month"),
    F.col("u.incoming_mou_per_month"),
    F.col("u.total_mou_per_month"),
    F.col("u.reporting_period").alias("usage_reporting_period")
)

(trai_context.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.trai_lsa_switching_usage_context"))

# National wireless trends
trends = spark.table("workspace.silver.trai_wireless_usage_trends")

(trends.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.trai_national_wireless_usage_trends"))

print("TRAI Gold tables completed!")


In [0]:

gold_tables = [
    "ibm_churn_overview",
    "ibm_churn_by_contract",
    "ibm_churn_by_payment_method",
    "ibm_churn_by_tenure",
    "ibm_churn_by_service",
    "trai_lsa_switching_usage_context",
    "trai_national_wireless_usage_trends"
]

for name in gold_tables:
    df = spark.table(f"workspace.gold.{name}")
    print(f"{name}: {df.count()} rows")

print("\nTotal Gold tables:", len(gold_tables))

spark.table("workspace.gold.ibm_churn_overview").show(truncate=False)


# Final data-quality validation

Check row counts, duplicate customer IDs, missing values, LSA mappings and the final Gold tables.

In [0]:

from pyspark.sql import functions as F

# 1. IBM customer checks
ibm = spark.table("workspace.silver.ibm_telco_customer_churn")

total = ibm.count()
unique_ids = ibm.select("CustomerID").distinct().count()

null_ids = ibm.filter(F.col("CustomerID").isNull()).count()
null_charges = ibm.filter(F.col("Total_Charges").isNull()).count()

churn_mismatches = ibm.filter(
    ((F.col("Churn_Label") == "Yes") & (F.col("Churn_Value") != 1)) |
    ((F.col("Churn_Label") == "No") & (F.col("Churn_Value") != 0))
).count()

print("IBM VALIDATION")
print("Total customers:", total)
print("Unique Customer IDs:", unique_ids)
print("Null Customer IDs:", null_ids)
print("Null Total Charges:", null_charges)
print("Churn mismatches:", churn_mismatches)

# 2. TRAI checks
checks = [
    ("trai_qos", 88),
    ("trai_mnp_service_area_status", 22),
    ("trai_usage_by_service_area", 22),
    ("trai_wireless_usage_trends", 5)
]

print("\nTRAI VALIDATION")

for name, expected in checks:
    actual = spark.table(f"workspace.silver.{name}").count()
    print(name, "| Expected:", expected, "| Actual:", actual)

# 3. LSA uniqueness
for name in [
    "trai_mnp_service_area_status",
    "trai_usage_by_service_area"
]:
    df = spark.table(f"workspace.silver.{name}")
    print(
        name,
        "| Unique LSAs:",
        df.select("canonical_lsa").distinct().count()
    )

# 4. Final Gold check
print("\nGOLD VALIDATION")
spark.table("workspace.gold.ibm_churn_overview").show()
spark.table("workspace.gold.trai_lsa_switching_usage_context").show(5)

print("\nVALIDATION CHECKS COMPLETED")
